# Formats: E2M1, E4M3 and NVFP4 into the bits of an f32

How the decoders build each f32 directly from its bits: the sign moves to bit 31, the exponent is re-biased (E' = e + 126 for E2M1, e + 120 for E4M3) and the mantissa is shifted to the top of the 23 mantissa bits.
An NVFP4 value is one E2M1 code (two per byte, low nibble first) times its E4M3 block scale (one per 16 values) times the F32 scale of the tensor.

In [ ]:
import re
import subprocess
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# the notebook sits in Claude Benchmark/<name>/, the repository two levels up
HERE = Path.cwd().resolve()
ROOT = HERE.parents[1]

MM = 1 / 25.4
INK, PAPER, GREY, SHADE = "#1f1d1a", "#faf8f3", "#77746e", "#d3cfc6"
BASELINE_GREY = "#a3a09a"
plt.rcParams.update({
    "font.family": "Liberation Sans", "font.size": 7, "axes.labelsize": 7,
    "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6.5,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.spines.top": False, "axes.spines.right": False,
})


def main_constant(name):
    """A constant of src/main.zig, such as `const MIN_BLOCKS_FOR_WORKERS = 16 * 1024;`, evaluated."""
    source = (ROOT / "src" / "main.zig").read_text(encoding="utf-8")
    expression = re.search(rf"^const {name} = ([0-9 *]+);", source, re.MULTILINE).group(1)
    return int(np.prod([int(factor) for factor in expression.split("*")]))


def run_benchmark(bench, csv):
    """Runs the benchmark (calls of bufferToWorker only) when its results are missing."""
    if not (HERE / csv).exists():
        subprocess.run(["zig", "run", "-OReleaseFast", "-mcpu=native", "--dep", "dequantizer",
                        f"-Mroot={HERE / bench}", f"-Mdequantizer={ROOT / 'src' / 'root.zig'}"],
                       cwd=ROOT, check=True)
    return pd.read_csv(HERE / csv)


In [ ]:
from matplotlib.colors import to_rgb
from matplotlib.patches import FancyArrowPatch, Rectangle

plt.rcParams.update({"mathtext.fontset": "custom", "mathtext.rm": "Liberation Sans",
                     "mathtext.it": "Liberation Sans:italic", "mathtext.bf": "Liberation Sans:bold"})
MONO = "Liberation Mono"

# one colour per field: brick for the sign, ink blue for the exponent, ochre for the mantissa
FIELD = {"s": "#b5533c", "e": "#34558b", "m": "#d1a042"}
NAME = {"s": "sign", "e": "exponent", "m": "mantissa"}
LINE = "#c9c3b8"

def tint(colour, amount):
    paper = np.array(to_rgb(PAPER))
    return tuple(paper + amount * (np.array(to_rgb(colour)) - paper))

fig, ax = plt.subplots(figsize=(175 * MM, 155 * MM))
ax.set_xlim(0, 17.5)
ax.set_ylim(-0.85, 14.45)
ax.set_aspect("equal")
ax.axis("off")

def source_bits(x, y, fields, cell=0.6, height=0.6, letters=True, top_bit=None):
    """A small format, fields = [(kind, count)] from the top bit down; returns the x span of each field."""
    spans, cursor = [], x
    for kind, count in fields:
        start = cursor
        for _ in range(count):
            ax.add_patch(Rectangle((cursor, y), cell, height, facecolor=tint(FIELD[kind], 0.25), edgecolor=PAPER, lw=0.8))
            if letters:
                ax.text(cursor + cell / 2, y + height / 2, kind, ha="center", va="center", fontsize=7.5,
                        color=FIELD[kind], style="italic")
            cursor += cell
        spans.append((kind, start, cursor))
    ax.add_patch(Rectangle((x, y), cursor - x, height, fill=False, edgecolor=INK, lw=0.7))
    if top_bit is not None:
        ax.text(x + cell / 2, y + height + 0.1, str(top_bit), ha="center", va="bottom", fontsize=5.5, color=GREY)
        ax.text(cursor - cell / 2, y + height + 0.1, "0", ha="center", va="bottom", fontsize=5.5, color=GREY)
    return spans

F32_X, F32_CELL = 6.6, 0.3

def f32_bits(y, styles, x=F32_X, cell=F32_CELL, height=0.42, labels=True):
    """The 32 bits of an f32, bit 31 on the left; styles[i] = (kind, strength), kind None for a bit left at 0."""
    for position, (kind, strength) in enumerate(styles):
        zero = kind is None
        ax.add_patch(Rectangle((x + position * cell, y), cell, height,
                               facecolor=PAPER if zero else tint(FIELD[kind], strength),
                               edgecolor=LINE if zero else PAPER, lw=0.5 if zero else 0.6))
    ax.add_patch(Rectangle((x, y), 32 * cell, height, fill=False, edgecolor=INK, lw=0.7))
    if labels:
        for position, label in ((0.0, "31"), (4.0, "30 – 23"), (20.0, "22 – 0")):
            ax.text(x + position * cell + cell / 2, y - 0.12, label, ha="center", va="top", fontsize=5.5, color=GREY)
        ax.text(x + 32 * cell + 0.2, y + height / 2, "f32", fontsize=7, color=GREY, va="center")

def centre(first, last):
    """x of the middle of f32 bits first..last (bit numbers, 31 on the left)."""
    return F32_X + ((31 - first) + (31 - last) + 1) / 2 * F32_CELL

def arrow(start, end, colour):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=6, color=colour, lw=0.8,
                                 shrinkA=2, shrinkB=2))

def code(x, y, lines, step=0.42):
    for index, line in enumerate(lines):
        ax.text(x, y - index * step, line, fontsize=6.3, color=INK, family=MONO, va="center")

# ---------- E2M1 (e2m1ToF32Simd): sign, exponent and mantissa are extracted, then placed ----------
y = 12.9
ax.text(0.5, y + 0.95, "E2M1", fontsize=9, fontweight="bold", color=INK, va="bottom")
spans = source_bits(0.5, y, [("s", 1), ("e", 2), ("m", 1)], top_bit=3)
target_y = y - 1.2
# (exponent + 126) << 23 fills the whole exponent, mantissa << 22 lands on bit 22 only
f32_bits(target_y, [("s", 0.25)] + [("e", 0.25)] * 8 + [("m", 0.25)] + [(None, 0)] * 22)
(_, s0, s1), (_, e0, e1), (_, m0, m1) = spans
arrow(((s0 + s1) / 2, y), (centre(31, 31), target_y + 0.42), FIELD["s"])
arrow(((e0 + e1) / 2, y), (centre(30, 23), target_y + 0.42), FIELD["e"])
arrow(((m0 + m1) / 2, y), (centre(22, 22), target_y + 0.42), FIELD["m"])
code(0.5, target_y - 0.75, [
    "sign      = bits >> 3",
    "exponent  = (bits >> 1) & 0b11",
    "mantissa  = bits & 1",
    "normal    = ((exponent + 126) << 23) | (mantissa << 22)",
    "subnormal = mantissa * (126 << 23)          // 0 or 0.5",
    "return @select(exponent == 0, subnormal, normal) | (sign << 31)",
], step=0.36)

# ---------- E4M3 (e4m3ToF32bits): exponent and mantissa move together with one shift ----------
y = 7.3
ax.text(0.5, y + 0.95, "E4M3", fontsize=9, fontweight="bold", color=INK, va="bottom")
spans = source_bits(0.5, y, [("s", 1), ("e", 4), ("m", 3)], top_bit=7)
target_y = y - 1.2
# (bits & 0x7F) << 20 puts e on bits 26-23 and m on bits 22-20; + (120 << 23) completes the exponent
f32_bits(target_y, [("s", 0.25)] + [("e", 0.10)] * 4 + [("e", 0.25)] * 4 + [("m", 0.25)] * 3 + [(None, 0)] * 20)
(_, s0, s1), (_, e0, e1), (_, m0, m1) = spans
arrow(((s0 + s1) / 2, y), (centre(31, 31), target_y + 0.42), FIELD["s"])
arrow(((e0 + m1) / 2, y), (centre(26, 20), target_y + 0.42), INK)
code(0.5, target_y - 0.75, [
    "sign      = (bits & 0x80) << 24",
    "expMant   = (bits & 0x7F) << 20",
    "normal    = expMant + (120 << 23)",
    "subnormal = f32(expMant + (121 << 23)) - 0x1p-6  // e == 0",
    "nan       = 0x7FC00000                     // (bits & 0x7F) == 0x7F",
    "return magnitude | sign",
], step=0.36)

# the bits both decoders build
ax.text(0.5, 3.05, "f32 bits = (s << 31) | (E' << 23) | M'", fontsize=7, color=INK, family=MONO, va="center")

# ---------- NVFP4 (decodeGroup): split the nibbles, then value = e2m1 * (e4m3 * scale2) ----------
y, cell = 1.0, 0.42
ax.text(0.5, y + 0.95, "NVFP4", fontsize=9, fontweight="bold", color=INK, va="bottom")
nibble = [("s", 1), ("e", 2), ("m", 1)]
source_bits(0.5, y, nibble, cell, 0.5)
source_bits(0.5 + 4 * cell, y, nibble, cell, 0.5)
ax.text(0.5 + 2 * cell, y - 0.12, ">> 4", ha="center", va="top", fontsize=5.8, color=GREY, family=MONO)
ax.text(0.5 + 6 * cell, y - 0.12, "& 0x0F", ha="center", va="top", fontsize=5.8, color=GREY, family=MONO)
ax.text(0.5 + 2 * cell, y + 0.62, "second", ha="center", va="bottom", fontsize=5.8, color=GREY)
ax.text(0.5 + 6 * cell, y + 0.62, "first", ha="center", va="bottom", fontsize=5.8, color=GREY)

ax.text(4.2, y + 0.25, r"$\times$", ha="center", va="center", fontsize=11, color=INK)
ax.text(4.65, y + 0.25, "(", ha="center", va="center", fontsize=14, color=INK)
source_bits(4.9, y, [("s", 1), ("e", 4), ("m", 3)], cell, 0.5)
ax.text(4.9 + 4 * cell, y - 0.12, "one per 16 values", ha="center", va="top", fontsize=5.8, color=GREY)
ax.text(8.65, y + 0.25, r"$\times$", ha="center", va="center", fontsize=11, color=INK)
f32_bits(y, [("s", 0.25)] + [("e", 0.25)] * 8 + [("m", 0.25)] * 23, x=9.05, cell=0.105, height=0.5, labels=False)
ax.text(9.05 + 32 * 0.105 / 2, y - 0.12, "scale2, one per tensor", ha="center", va="top", fontsize=5.8, color=GREY)
ax.text(12.6, y + 0.25, ")", ha="center", va="center", fontsize=14, color=INK)
ax.text(13.05, y + 0.25, "=", ha="center", va="center", fontsize=11, color=INK)
f32_bits(y, [("s", 0.25)] + [("e", 0.25)] * 8 + [("m", 0.25)] * 23, x=13.45, cell=0.105, height=0.5, labels=False)
ax.text(13.45 + 32 * 0.105 / 2, y - 0.12, "f32 value", ha="center", va="top", fontsize=5.8, color=GREY)

code(0.5, -0.2, [
    "factors = e4m3ToF32bits(16, scales) * scale2s     // one factor per block",
    "value   = e2m1ToF32Simd(codes) * factor",
], step=0.38)

# which colour is which field
x = 11.0
for kind in "sem":
    ax.add_patch(Rectangle((x, 14.0), 0.32, 0.32, facecolor=tint(FIELD[kind], 0.25), edgecolor=FIELD[kind], lw=0.8))
    label = ax.text(x + 0.45, 14.16, NAME[kind], fontsize=7, color=INK, va="center")
    x += 0.45 + 0.17 * len(NAME[kind]) + 0.55

fig.savefig(HERE / "formats.png", dpi=300, bbox_inches="tight", pad_inches=0.2)